# Opcion HistGradientBoosting

In [1]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
#codigo mas ligero
ruta = 'work/autotec/app/datos_autotec.csv'
df = pd.read_csv(ruta)

objetivo = 'precio'
df = df.dropna(subset=[objetivo]).copy()

X = df.drop(columns=[objetivo])
y = df[objetivo]

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ],
    sparse_threshold=0
)

modelo = HistGradientBoostingRegressor(
    random_state=42,
    max_iter=300,
    learning_rate=0.05,
    max_depth=8,
    min_samples_leaf=20
)

pipe = Pipeline([
    ('prep', preprocessor),
    ('model', modelo)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print('=' * 50)
print('EVALUACIÓN MODELO HISTGRADIENTBOOSTING')
print('=' * 50)
print(f'R2: {r2:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'MAE: {mae:.4f}')
print('=' * 50)

resultados = pd.DataFrame({'y_real': y_test.values, 'y_predicho': y_pred})
resultados.to_csv('resultados_GradientBoosting.csv', index=False)

metricas = pd.DataFrame([{
    'modelo': 'HistGradientBoostingRegressor',
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'parametros': '{"max_iter":300, "learning_rate":0.05, "max_depth":8, "min_samples_leaf":20}'
}])
metricas.to_csv('metricas_histgb_rapido.csv', index=False)

joblib.dump(pipe, 'modelo_histgb_rapido_autotec.joblib')

EVALUACIÓN MODELO HISTGRADIENTBOOSTING
R2: 0.6848
RMSE: 5372485.3684
MAE: 3390798.4244


['modelo_histgb_rapido_autotec.joblib']

## Resultado:
El modelo HistGradientBoostingRegressor alcanzó un R2 2
  de 0.6848, lo que indica una capacidad explicativa moderadamente buena sobre la variación del precio de los vehículos usados. El MAE de 3,390,798.42 y el RMSE de 5,372,485.37 muestran que aún existen errores relevantes en la predicción, especialmente en casos con valores extremos o vehículos con características poco frecuentes. Aun así, el desempeño supera al modelo lineal, lo que confirma que la relación entre las variables explicativas y el precio no es completamente lineal.

## Comparación entre modelos
Se evaluaron tres enfoques para predecir el precio de los vehículos usados: Regresión Lineal, Random Forest y HistGradientBoosting. La regresión lineal obtuvo un desempeño menor, con un \(R^2 = 0.5831\), lo que indica que su capacidad para explicar la variación del precio es limitada frente a modelos más complejos. En cambio, Random Forest mostró un mejor ajuste y fue más preciso para capturar relaciones no lineales entre las variables explicativas y el precio. Por su parte, HistGradientBoosting alcanzó un \(R^2 = 0.6848\), mejorando a la regresión lineal, aunque todavía sin superar completamente al mejor modelo de árboles probado. En términos generales, los resultados confirman que el comportamiento del precio en el mercado automotriz no es estrictamente lineal, por lo que los modelos basados en árboles y boosting resultan más adecuados para este problema.